# Settings

In [6]:
#!/opt/homebrew/bin/python3 -m pip install ipykernel -U --user --force-reinstall

# this notebook is from Thu's M2 mac, so I will setup based on mac installation
!pip3 install selenium

# download this zip chromedriver directory to your local machine (choose your adaptation): https://getwebdriver.com/chromedriver#stable
# mine: https://storage.googleapis.com/chrome-for-testing-public/126.0.6478.182/mac-arm64/chrome-mac-arm64.zip

# !mv ~/Downloads/chrome-mac-arm64 /usr/local/bin

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time

In [8]:
# chrome driver setup
chrome_options = Options()
chrome_options.add_argument('--headless')  # Runs Chrome in headless mode.
chrome_options.add_argument('--no-sandbox')  # Bypass OS security model
chrome_options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 20)  # wait for 20s

# Actions

* Login to the E-learning SSO authentication

In [9]:
driver.get('https://sso.hcmut.edu.vn/cas/login?service=https%3A%2F%2Fe-learning.hcmut.edu.vn%2Flogin%2Findex.php%3FauthCAS%3DCAS')
username = driver.find_element(By.ID, 'username')
password = driver.find_element(By.ID, 'password')
username.send_keys('010344')
password.send_keys('010344')
password.send_keys(Keys.RETURN)

* Select the URL that the user wants to go inside (you can use `element_to_be_clickable` function or recognize the tab existence by using `presence_of_element_located` function and then execute the driver).

In [14]:
def find_dom_link(tested_url, accessed_url):
    try:
        driver.get(tested_url)
        link = driver.find_element_by_xpath('//a[@href="f{accessed_url}"]')
        print("Link exists in the DOM.")
    except NoSuchElementException:
        print("Link does not exist in the DOM.")

In [ ]:
if find_dom_link('https://e-learning.hcmut.edu.vn/course/view.php?id=108885', )

In [10]:
my_courses_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[@href='https://e-learning.hcmut.edu.vn/my/courses.php']")))
my_courses_link.click()

In [11]:
dsa_link = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[@href='https://e-learning.hcmut.edu.vn/course/view.php?id=108885']"))) # DT
dsa_link.click()

In [12]:
dsa_oop_link = wait.until(EC.presence_of_element_located((By.XPATH, "//a[@href='https://e-learning.hcmut.edu.vn/mod/quiz/view.php?id=188779']")))
driver.execute_script("arguments[0].click();", dsa_oop_link)

In [ ]:
dsa_oop_result_link = wait.until(EC.presence_of_element_located((By.XPATH, "//a[@href='https://e-learning.hcmut.edu.vn/mod/quiz/report.php?id=188779&mode=overview']")))
driver.execute_script("arguments[0].click();", dsa_oop_result_link)

In [ ]:
driver.get('https://e-learning.hcmut.edu.vn/mod/quiz/report.php?id=188779&mode=overview')

* Open the inspect and see the structure and location of each student's information (it's in the region-main).
* We will create a dictionary storing the lab's name, list of questions and student's answers.
* Each question contains a plain text and a table represents a sample testcase. We have to find a tab table and retrieve each row.
* After retrieving all information needed, we append to each corresponding category.

In [ ]:
wait.until(EC.visibility_of_element_located((By.ID, 'region-main')))
lab_name = driver.title.split(":")[0].strip()

data = {
    'lab_name': lab_name,
    'list_questions': [],
    'student_answers': []
}

wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, 'table')))
student_rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')

for row in student_rows[:-4]:
    # if row.find_element(By.CSS_SELECTOR, 'td.cell.c2.bold') is not None:
    student_info = row.find_element(By.CSS_SELECTOR, 'td.cell.c2.bold')
    student_name = student_info.find_element(By.TAG_NAME, 'a').text.strip()
    student_id = row.find_element(By.CSS_SELECTOR, 'td.cell.c3').text.strip()
    review_link = student_info.find_element(By.CSS_SELECTOR, 'a.reviewlink').get_attribute('href')

    data['student_answers'].append({
        'name': student_name,
        'id': student_id,
        'review_link': review_link
    })

In [ ]:
driver.get(data['student_answers'][0]['review_link'])
questions = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.que.coderunner')))
list_questions = []

for question in questions:
    question_text = " ".join(question.find_element(By.CSS_SELECTOR, 'div.content div.formulation').text.split())
    coderunner_examples_div = question.find_element(By.CSS_SELECTOR, 'div.coderunner-examples')
    expected_output_table = coderunner_examples_div.find_element(By.CSS_SELECTOR, 'table.coderunnerexamples')

    rows = expected_output_table.find_elements(By.CSS_SELECTOR, 'tbody tr')
    expected_outputs = []
    for row in rows:
        test_cell = row.find_element(By.CSS_SELECTOR, 'td.cell.c0 pre.tablecell').text
        result_cell = row.find_element(By.CSS_SELECTOR, 'td.cell.c1 pre.tablecell').text
        expected_outputs.append({'test': test_cell, 'result': result_cell})

    list_questions.append({
        'question': question_text,
        'expected_outputs': expected_outputs
    })

data['list_questions'] = list_questions

In [ ]:
for record in data['student_answers']:
    driver.get(record['review_link'])
    attempt_data = []

    wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'responsehistoryheader')))
    history_headers = driver.find_elements(By.XPATH, "//h4[contains(text(), 'Response history')]")

    for index, header in enumerate(history_headers):
        table = header.find_element(By.XPATH, "following-sibling::div//table[@class='generaltable']")
        rows = table.find_elements(By.CSS_SELECTOR, 'tbody tr')
        table_data = {
            'question': f'Question {index+1}',
            'results': []
        }
        for row in rows:
            cells = row.find_elements(By.CSS_SELECTOR, 'td')
            if len(cells) >= 5:
                step = cells[0].text
                time = cells[1].text
                action = cells[2].text
                state = cells[3].text
                marks = cells[4].text

                table_data['results'].append({
                    'step': step,
                    'time': time,
                    'action': action,
                    'state': state,
                    'marks': marks
                })

        attempt_data.append(table_data)

    record['response_history'] = attempt_data

In [ ]:
data

{'lab_name': 'OOP Review',
 'list_questions': [{'question': 'Question text In the coordinate plane, we have class Point to store a point with it\'s x-y coordinate. Your task in this exercise is to implement functions marked with /* * STUDENT ANSWER */. Note: For exercises in Week 1, we have #include <bits/stdc++.h> and using namespace std; For example: Test Result Point A(2, 3); cout << A.getX() << " " << A.getY(); 2 3 Point A(2, 3); Point B(1, 1); cout << pow(A.distanceToPoint(B), 2); 5 Answer:(penalty regime: 0 %) Ace editor not ready. Perhaps reload page? Falling back to raw text area. class Point { private: double x, y; public: Point() { /* * STUDENT ANSWER * TODO: set zero x-y coordinate */ this->x = 0; this->y = 0; } Point(double x, double y) { /* * STUDENT ANSWER */ this->x = x; this->y = y; } void setX(double x) { /* * STUDENT ANSWER */ this->x = x; } void setY(double y) { /* * STUDENT ANSWER */ this->y = y; } double getX() const { /* * STUDENT ANSWER */ return this->x; } doubl

In [ ]:
import json

with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)